# 03 — Audio Feature Extraction & Emotion Classification

## Research Question
**Can we classify emotional sentiment (positive / negative / neutral) from speech audio features alone?**

## Approach
Every audio recording is a waveform — a sequence of numbers representing air pressure over time.
We extract **numerical features** from each waveform:

| Feature | What it captures | # values |
|---------|-----------------|---------|
| MFCCs (mean+std) | Vocal tract shape | 26 (13×2) |
| Pitch F0 (mean+std) | Voice pitch | 2 |
| RMS (mean+std) | Loudness | 2 |
| Spectral Centroid (mean+std) | Brightness | 2 |
| Zero Crossing Rate (mean+std) | Noisiness | 2 |
| **Total** | | **34 features** |

These 34 numbers become one row in our feature matrix.
We then train a **Random Forest classifier** and evaluate it with **LOSO cross-validation**.


## 0. Setup

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, matplotlib.gridspec as gridspec
import librosa, librosa.display
import seaborn as sns
from datasets import load_dataset
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import LabelEncoder

plt.rcParams.update({"figure.dpi":120,"font.family":"sans-serif"})
FIG_DIR = Path("../reports/figures"); FIG_DIR.mkdir(parents=True, exist_ok=True)

print("Loading IEMOCAP dataset...")
ds = load_dataset("AbstractTTS/IEMOCAP", split="train")
print(f"Total recordings: {len(ds):,}")

SENTIMENT_MAP = {
    "happy":"positive","excited":"positive",
    "neutral":"neutral","surprise":"neutral",
    "fear":"negative","frustrated":"negative","sad":"negative","angry":"negative",
}
EMO_COLORS = {"angry":"#e74c3c","happy":"#f39c12","neutral":"#3498db","sad":"#9b59b6","frustrated":"#c0392b","excited":"#e67e22"}
SENT_COLORS = {"positive":"#f39c12","neutral":"#3498db","negative":"#e74c3c"}
print("Ready ✅")


## 1. What Does a Speech Waveform Look Like?

Before extracting features we visualize **what the audio signal actually is**.

- **Waveform**: raw amplitude over time
- **Spectrogram**: which frequencies are active at each moment
- **MFCCs**: compact representation of the spectral shape (13 coefficients per frame)


In [ ]:
sample = ds[10]
y  = np.array(sample["audio"]["array"], dtype=np.float32)
sr = sample["audio"]["sampling_rate"]
emotion   = sample["major_emotion"]
sentiment = SENTIMENT_MAP.get(emotion, "unknown")

print(f"Emotion: {emotion}  →  Sentiment: {sentiment}")
print(f"Duration: {len(y)/sr:.2f}s  |  Sample rate: {sr} Hz  |  Samples: {len(y):,}")

fig = plt.figure(figsize=(14, 9))
gs  = gridspec.GridSpec(3, 1, hspace=0.5)

ax1 = fig.add_subplot(gs[0])
librosa.display.waveshow(y, sr=sr, ax=ax1, color="#2980b9", alpha=0.8)
ax1.set_title(f"Waveform — Emotion: {emotion}", fontweight="bold")

ax2 = fig.add_subplot(gs[1])
D   = librosa.amplitude_to_db(np.abs(librosa.stft(y)), ref=np.max)
img = librosa.display.specshow(D, sr=sr, x_axis="time", y_axis="hz", ax=ax2, cmap="magma")
ax2.set_title("Spectrogram — Frequency content over time", fontweight="bold")
fig.colorbar(img, ax=ax2, format="%+2.0f dB")

ax3 = fig.add_subplot(gs[2])
mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
img2  = librosa.display.specshow(mfccs, sr=sr, x_axis="time", ax=ax3, cmap="coolwarm")
ax3.set_title("MFCCs (13 coefficients) — Compact vocal tract representation", fontweight="bold")
fig.colorbar(img2, ax=ax3)

plt.savefig(FIG_DIR/"A1_single_audio.png", bbox_inches="tight"); plt.show()
print("\n📌 Key observations:")
print("  Waveform: amplitude ~ loudness. Angry = wide, sad = narrow")
print("  Spectrogram: high-pitched voice = energy at high frequencies")
print("  MFCCs: 13 numbers per frame capture the 'shape' of the spectrum")


## 2. Comparing Waveforms Across Emotions

In [ ]:
TARGET = ["angry","happy","neutral","sad"]
samples_by_emo = {}
for item in ds:
    emo = item["major_emotion"]
    if emo in TARGET and emo not in samples_by_emo:
        samples_by_emo[emo] = item
    if len(samples_by_emo) == len(TARGET): break

fig, axes = plt.subplots(len(TARGET), 2, figsize=(16, 10))
for row, emo in enumerate(TARGET):
    item   = samples_by_emo[emo]
    y_     = np.array(item["audio"]["array"], dtype=np.float32)
    sr_    = item["audio"]["sampling_rate"]
    color  = EMO_COLORS.get(emo, "#555")
    librosa.display.waveshow(y_, sr=sr_, ax=axes[row,0], color=color, alpha=0.75)
    axes[row,0].set_title(f"{emo.upper()} — Waveform", fontweight="bold", color=color)
    S    = librosa.feature.melspectrogram(y=y_, sr=sr_, n_mels=64)
    S_db = librosa.power_to_db(S, ref=np.max)
    librosa.display.specshow(S_db, sr=sr_, x_axis="time", y_axis="mel",
                             ax=axes[row,1], cmap="magma")
    axes[row,1].set_title(f"{emo.upper()} — Mel Spectrogram", fontweight="bold", color=color)
plt.suptitle("Emotion Comparison — Waveform & Mel Spectrogram", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(FIG_DIR/"A2_emotion_comparison.png", bbox_inches="tight"); plt.show()
print("\n📌 Angry: wide waveform, rich high-frequency content")
print("   Sad:   narrow waveform, mostly low frequencies")
print("   Happy: energetic but less extreme than angry")
print("   Neutral: clean, stable, mid-range frequencies")


## 3. Feature Extraction Function

We extract 34 features per recording.

**Why mean + std?**
Each feature varies frame-by-frame. We summarize the entire recording
with mean (overall value) and standard deviation (how much it varies).


In [ ]:
def extract_features(y: np.ndarray, sr: int) -> dict:
    """Extract 34 acoustic features from a single audio recording."""
    y = y.astype(np.float32)
    feats = {}

    # MFCCs — 13 coefficients × (mean + std) = 26 values
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    for i in range(13):
        feats[f"mfcc_{i+1}_mean"] = float(np.mean(mfcc[i]))
        feats[f"mfcc_{i+1}_std"]  = float(np.std(mfcc[i]))

    # Pitch (F0) — fundamental frequency
    f0, _, _ = librosa.pyin(y, fmin=50, fmax=500, sr=sr)
    f0c = f0[~np.isnan(f0)] if f0 is not None else np.array([0.0])
    feats["pitch_mean"] = float(np.mean(f0c)) if len(f0c) > 0 else 0.0
    feats["pitch_std"]  = float(np.std(f0c))  if len(f0c) > 0 else 0.0

    # RMS energy
    rms = librosa.feature.rms(y=y)[0]
    feats["rms_mean"] = float(np.mean(rms)); feats["rms_std"] = float(np.std(rms))

    # Spectral Centroid — brightness
    sc = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
    feats["sc_mean"] = float(np.mean(sc)); feats["sc_std"] = float(np.std(sc))

    # Zero Crossing Rate — noisiness
    zcr = librosa.feature.zero_crossing_rate(y)[0]
    feats["zcr_mean"] = float(np.mean(zcr)); feats["zcr_std"] = float(np.std(zcr))

    return feats

# Quick test
test = extract_features(np.array(ds[0]["audio"]["array"]), ds[0]["audio"]["sampling_rate"])
print(f"Features extracted per recording: {len(test)}")
for k, v in list(test.items())[:5]:
    print(f"  {k}: {v:.4f}")
print("Feature extraction function ready ✅")


## 4. Extract Features from All Recordings

We process every valid recording in the dataset.
This step takes ~10–15 minutes (librosa computes pitch for each file).


In [ ]:
valid_emotions = set(SENTIMENT_MAP.keys())
rows, errors = [], 0

print(f"Extracting features from {len(ds):,} recordings...")
for i, item in enumerate(ds):
    emo = item.get("major_emotion", "")
    if emo not in valid_emotions:
        continue
    try:
        y_   = np.array(item["audio"]["array"], dtype=np.float32)
        sr_  = item["audio"]["sampling_rate"]
        if len(y_) < sr_ * 0.3:   # skip clips shorter than 0.3s
            continue
        feats = extract_features(y_, sr_)
        feats["emotion"]   = emo
        feats["sentiment"] = SENTIMENT_MAP[emo]
        # Extract session ID from filename
        fname = item.get("file", "")
        feats["session"] = fname.split("/")[0] if "/" in fname else fname[:5]
        rows.append(feats)
    except Exception:
        errors += 1
    if (i+1) % 500 == 0:
        print(f"  Processed {i+1:,} / {len(ds):,} | Valid so far: {len(rows):,}")

df = pd.DataFrame(rows)
print(f"\n✅ Done! Valid recordings: {len(df):,}  |  Errors: {errors}")
print("\nClass distribution:")
print(df["sentiment"].value_counts().to_string())

df.to_csv("../data/audio_features.csv", index=False)
print("\nSaved: data/audio_features.csv")


## 5. Feature Distributions by Sentiment Group

**Key question:** Do audio features differ between sentiment classes?
If distributions overlap completely → feature is not useful for classification.
If distributions are separated → feature helps the model.


In [ ]:
show_feats = ["mfcc_1_mean","mfcc_2_mean","pitch_mean","rms_mean","sc_mean","zcr_mean"]
show_feats = [f for f in show_feats if f in df.columns]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, feat in zip(axes.flat, show_feats):
    for sent, color in SENT_COLORS.items():
        data = df[df["sentiment"]==sent][feat].dropna()
        data.plot(kind="kde", ax=ax, label=sent, color=color, linewidth=2.2)
    ax.set_title(feat, fontweight="bold"); ax.set_xlabel("Value"); ax.legend(fontsize=8)

plt.suptitle("Feature Distributions by Sentiment\n(Separated curves → feature is discriminative)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(FIG_DIR/"A3_feature_distributions.png", bbox_inches="tight"); plt.show()

print("\nMean feature values by sentiment group:")
print(df.groupby("sentiment")[show_feats].mean().round(3).to_string())


## 6. Model Training — Random Forest with LOSO Cross-Validation

### Why LOSO?
IEMOCAP has 5 sessions, each with different speakers.
In each LOSO round: train on 4 sessions, test on 1.
This prevents speaker leakage (model memorising a specific voice).

### Metric: UAR (Unweighted Average Recall)
We use UAR instead of accuracy because classes are imbalanced.
UAR = mean of per-class recall. Random baseline = 0.333 for 3 classes.


In [ ]:
feature_cols = [c for c in df.columns if c not in ["emotion","sentiment","session"]]
X      = df[feature_cols].fillna(0).values
y_sent = df["sentiment"].values
df["session_id"] = df["session"].str.extract(r"(Ses\d+|ses\d+|S\d+)")[0].fillna(df["session"].str[:5])
sessions = sorted(df["session_id"].unique())
print(f"Sessions found: {sessions}")

all_preds, all_true, session_scores = [], [], []
for test_sess in sessions:
    train_mask = df["session_id"] != test_sess
    test_mask  = df["session_id"] == test_sess
    if test_mask.sum() == 0: continue
    X_train,y_train = X[train_mask], y_sent[train_mask]
    X_test, y_test  = X[test_mask],  y_sent[test_mask]
    clf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
    clf.fit(X_train, y_train)
    preds = clf.predict(X_test)
    all_preds.extend(preds); all_true.extend(y_test)
    cr  = classification_report(y_test, preds, output_dict=True, zero_division=0)
    uar = np.mean([cr[c]["recall"] for c in ["positive","neutral","negative"] if c in cr])
    session_scores.append((test_sess, uar, test_mask.sum()))
    print(f"  {test_sess}: UAR={uar:.3f}  (n={test_mask.sum()})")

mean_uar = np.mean([s[1] for s in session_scores])
print(f"\n=== Mean LOSO UAR: {mean_uar:.3f} ===")
print(f"    Random baseline: 0.333")
print(f"    Improvement over random: {(mean_uar-0.333)/0.333*100:.1f}%")


## 7. Results — Confusion Matrix & Feature Importance

In [ ]:
classes = ["negative","neutral","positive"]
cm   = confusion_matrix(all_true, all_preds, labels=classes)
disp = ConfusionMatrixDisplay(cm, display_labels=classes)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
disp.plot(ax=axes[0], cmap="Blues", colorbar=False)
axes[0].set_title("Confusion Matrix (LOSO)\nRow=True, Column=Predicted", fontweight="bold")

clf_full = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
clf_full.fit(X, y_sent)
importances = pd.Series(clf_full.feature_importances_, index=feature_cols)
top15 = importances.nlargest(15)
axes[1].barh(top15.index[::-1], top15.values[::-1], color="#2980b9")
axes[1].set_xlabel("Feature Importance"); axes[1].set_title("Top 15 Most Important Features", fontweight="bold")
plt.tight_layout()
plt.savefig(FIG_DIR/"A4_results.png", bbox_inches="tight"); plt.show()

print("\nClassification Report (all LOSO folds):")
print(classification_report(all_true, all_preds, target_names=classes, zero_division=0))
print(f"\n📌 Most important feature: {importances.idxmax()}")


## 8. Conclusions

### What We Found

1. **Feature extraction**: 34 acoustic features extracted per recording (MFCCs, pitch, RMS, etc.)
2. **Visual separation**: Feature distributions differ across sentiment classes → audio carries emotion signal
3. **Random Forest UAR**: Beats random baseline (0.333) → audio features contain sentiment information
4. **Key features**: The first few MFCCs contribute most to classification

### Next Steps
- **More MFCCs**: Use 20-40 coefficients instead of 13
- **Better models**: SVM with RBF kernel, MLP, XGBoost
- **Deep learning**: CNN on mel-spectrogram, wav2vec2 fine-tuning
- **Text fusion**: Combine audio features with transcription embeddings
